# S&P 500 Statistical Diagnostics & Stylized Facts

This notebook extends the exploratory data analysis phase by formally testing the statistical properties of S&P 500 daily log returns.

The objective is to determine whether the return series satisfies the assumptions required for time series modeling and to identify key characteristics commonly observed in financial markets.

The analysis focuses on:

- Distribution properties
- Tail behavior
- Autocorrelation structure
- Volatility persistence
- Heteroskedasticity
- White noise behavior

These diagnostics provide the statistical foundation for later forecasting, volatility modeling, and anomaly detection phases.

In [1]:
# Import needed libraries
import pandas as pd 
import numpy as np 
# Setting Plotly backend plotting for pandas 
pd.options.plotting.backend = 'plotly'
import plotly.io as pio
pio.templates.default = 'plotly_dark'
import plotly.graph_objects as go
import plotly.express as px

import scipy.stats as stats
from scipy.stats import norm
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox

## Import Required Libraries

The following libraries are used for:

- Statistical testing
- Distribution analysis
- Autocorrelation diagnostics
- Volatility diagnostics
- Visualization

These tools allow us to formally evaluate the statistical behavior of financial return series.

## Load Data From CSV

In [2]:
df = pd.read_parquet('../data/sp500_cleaned.parquet').copy()
# preview Dataset
df.head()

Price,Open,High,Low,Close,Volume,log_returns
Date,,,,,,
2000-01-04,1455.219971,1455.219971,1397.430054,1399.420044,1009000000,-0.039099
2000-01-05,1399.420044,1413.270020,1377.680054,1402.109985,1085500000,0.001920
2000-01-06,1402.109985,1411.900024,1392.099976,1403.449951,1092300000,0.000955
2000-01-07,1403.449951,1441.469971,1400.729980,1441.469971,1225200000,0.026730
2000-01-10,1441.469971,1464.359985,1441.469971,1457.599976,1064800000,0.011128


## Load Cleaned Dataset

The dataset used in this notebook was cleaned and validated during the EDA phase.

This includes:

- Removal of anomalous observations
- Validation of trading dates
- Log return calculation
- Consistent datetime indexing

Using a pre-cleaned dataset ensures reproducibility and prevents duplication of preprocessing steps across notebooks.

In [3]:
# Dataset information
df.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 6626 entries, 2000-01-04 to 2026-05-11
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Open         6626 non-null   float64
 1   High         6626 non-null   float64
 2   Low          6626 non-null   float64
 3   Close        6626 non-null   float64
 4   Volume       6626 non-null   int64  
 5   log_returns  6626 non-null   float64
dtypes: float64(5), int64(1)
memory usage: 362.4 KB


In [4]:
# Check Index Type
type(df.index)

pandas.DatetimeIndex

In [5]:
# Summary Statistics
df.describe()

Price,Open,High,Low,Close,Volume,log_returns
count,6626.000000,6626.000000,6626.000000,6626.000000,6.626000e+03,6626.000000
mean,2330.478806,2343.720599,2316.252649,2330.873291,3.448175e+09,0.000246
std,1539.016777,1546.145242,1531.428490,1539.443925,1.525790e+09,0.012176
min,679.280029,695.270020,666.789978,676.530029,3.560700e+08,-0.127652
25%,1213.117493,1220.277496,1205.482483,1213.540039,2.344672e+09,-0.004730
50%,1552.229980,1557.010010,1545.515015,1552.419983,3.544525e+09,0.000642
75%,2932.627441,2946.345032,2919.170105,2938.012573,4.293225e+09,0.005883
max,7385.310059,7428.970215,7384.200195,7412.839844,1.145623e+10,0.109572


## Dataset Validation

Before performing statistical diagnostics, the dataset structure is briefly validated to confirm:

- Correct datetime indexing
- Presence of return series
- Numerical consistency
- Expected observation count

This step ensures the dataset is properly prepared for statistical testing.

# Distribution Diagnostics 

## 1) Skewness & Kurtosis

In [6]:
skewness = df['log_returns'].skew()
kurtosis = df['log_returns'].kurtosis()

print(f'Skewness: {skewness:.4f}')
print(f'Kurtosis: {kurtosis:.4f}')

Skewness: -0.3479
Kurtosis: 10.6424


## 📊 Interpretation of Skewness & Kurtosis Results

### Results

- **Skewness:** **-0.3479**  
- **Excess Kurtosis:** **10.6424**


### Interpretation of Skewness

The negative skewness indicates that the return distribution is slightly tilted toward larger downside moves.

This means:

- Negative returns tend to be more extreme than positive returns  
- Market declines are often sharper and more sudden than rallies  
- Downside tail risk is more pronounced  

This behavior is common in equity markets, where panic selling and systemic shocks can produce rapid drawdowns.


### Interpretation of Kurtosis

The excess kurtosis value of **10.6424** is substantially above zero.

The high kurtosis observed here indicates:

- The distribution contains **fat tails**
- Extreme returns occur far more frequently than predicted by Gaussian assumptions
- Market crashes and large rallies are statistically more common than standard models would expect


Financial return series often violate normal distribution assumptions due to:

- Fat tails  
- Extreme market events  
- Time-varying volatility  

These characteristics motivate the use of more advanced volatility and risk models in later stages of the project.

### Key Insight

> Extreme market movements occur more frequently than predicted by standard normal distribution assumptions.

# Q-Q Plot (Quantile-Quantile Plot)

In [7]:

# Generate Q-Q plot data
(osm, osr), (slope, intercept, r) = stats.probplot(
    df['log_returns'],
    dist='norm'
)

# Create Plotly figure
fig = go.Figure()

# Sample quantiles
fig.add_trace(
    go.Scatter(
        x=osm,
        y=osr,
        mode='markers',
        name='Observed Returns',
        marker=dict(size=4)
    )
)

# Reference normal line
fig.add_trace(
    go.Scatter(
        x=osm,
        y=slope * np.array(osm) + intercept,
        mode='lines',
        name='Normal Distribution Reference'
    )
)

# Layout
fig.update_layout(
    title='Q-Q Plot: S&P 500 Log Returns vs Normal Distribution',
    xaxis_title='Theoretical Quantiles',
    yaxis_title='Observed Quantiles',
    template='plotly_dark',
    height=700,
    width=900,
    legend_title=''
)

# Show plot
fig.show()

# Save figure
fig.write_image(
    "../reports/figures/sp500_qq_plot.png",
    width=1200,
    height=900,
    scale=2
)

# The Jarque-Bera Test

In [ ]:
# Unpack the test results for clarity
jb_stat, p_value = stats.jarque_bera(df['log_returns'])

# Print results
print(f"Jarque-Bera Statistic: {jb_stat:,.2f}")
print(f"P-Value:               {p_value:.4f}")

# Add a simple logic check for the Null Hypothesis
alpha = 0.05
if p_value < alpha:
    print("\nResult: Reject H0 - The distribution is NOT normal (Fat Tails confirmed).")
else:
    print("\nResult: Fail to reject H0 - The distribution appears normal.")

Jarque-Bera Statistic: 31,350.69
P-Value:               0.0000

Result: Reject H0 - The distribution is NOT normal (Fat Tails confirmed).


### 📊 Jarque-Bera Normality Test

The Jarque-Bera test evaluates whether a dataset’s skewness and kurtosis are consistent with a normal distribution.

### **Results**

- **Jarque-Bera Statistic:** 31,350.69  
- **P-Value:** < 0.05  

### **Interpretation**

The p-value is significantly below the 5% significance level, so we reject the null hypothesis of normality.

This confirms that S&P 500 daily log returns do **not** follow a Gaussian (normal) distribution.

The result is consistent with earlier findings:

- Presence of **fat tails**
- Extreme market movements
- Volatility clustering
- Non-constant variance (heteroskedasticity)

### **Why This Matters in Finance**

Classical financial models often assume returns are normally distributed.

Under normality assumptions:

- Extreme losses are expected to occur very rarely
- Risk metrics may underestimate true downside exposure

However, real financial markets experience large shocks more frequently than a normal distribution predicts.

As a result:

- Tail risk can be underestimated
- Portfolio risk forecasts become less reliable during crises
- More advanced volatility models are required

### **Modeling Implication**

These findings provide statistical justification for moving toward:

- **ARCH/GARCH volatility models**
- Heavy-tailed distributions 
- Regime-switching approaches
- Stochastic volatility frameworks

These models are specifically designed to handle:
- Time-varying volatility
- Volatility clustering
- Fat-tailed return distributions


### **Key Insight**

> Financial return distributions are not well described by simple Gaussian assumptions.  
> Robust risk modeling requires methods capable of capturing tail risk and changing market volatility.

# Serial dependence diagnostics